# Captum for LLM explainability in unit testing (1)

This notebook aims to explore how LLMs understand code to write unit tests. The goal is to show how models understand the input prompt via feature importance using Captum library.

The chosen model to study is Qwen2.5-Coder-3B. 

We will work with the original model, without quantization nor fine-tuning.

The main method to obtain explainability is feature ablation, which computes attribution score to each token(word) in the input prompt. The meaning of this score is to show if a token highly contributes to the predicted output. Higher score means more contribution.

We will use the method on multiple python functions, from basic functions such as calculating sum of two numbers to more advanced and complicated functions.

In [1]:
import warnings
import tokenizers
from transformers import AutoModelForCausalLM, AutoTokenizer,AutoConfig
from datasets import load_dataset

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

import evaluate
from transformers import TrainingArguments, Trainer

import sys 
print(sys.executable)
import torch
print(torch)

import accelerate
import peft
import transformers

import psutil
import platform

import captum
from captum.attr import IntegratedGradients, Occlusion, LayerGradCam, LayerAttribution
from captum.attr import visualization as viz

print("===== SYSTEM INFO =====")
print("OS:", platform.system(), platform.release())
print("Processor:", platform.processor())
print("CPU cores:", psutil.cpu_count())

ram = psutil.virtual_memory()
print("RAM:", round(ram.total / (1024**3), 2), "GB")

print("\n===== GPU INFO =====")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

print("\n===== STACK ======")
print("torch version: ", torch.__version__)
print("accelerate version: ", accelerate.__version__)
print("peft version: ", peft.__version__)
print("transformers version: ", transformers.__version__)
print("captum version: ", captum.__version__)

/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/vasco/nguyehmk/miniconda3/envs/llm/bin/python
<module 'torch' from '/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/torch/__init__.py'>
===== SYSTEM INFO =====
OS: Linux 6.1.0-48-amd64
Processor: 
CPU cores: 32
RAM: 251.39 GB

===== GPU INFO =====
CUDA available: True
GPU: NVIDIA GeForce GTX 1080 Ti
VRAM: 10.91 GB

===== STACK ======
torch version:  2.0.1+cu117
accelerate version:  0.28.0
peft version:  0.10.0
transformers version:  4.38.2
captum version:  0.6.0


In [2]:
from captum.attr import (
    FeatureAblation, 
    ShapleyValues,
    LayerIntegratedGradients, 
)

# Ignore warnings due to transformers library
warnings.filterwarnings("ignore", ".*past_key_values.*")
warnings.filterwarnings("ignore", ".*Skipping this token.*")


In [3]:
# Load model directly
torch.cuda.empty_cache()

# Check device availability
device = torch.device("cuda")
print("Model device:", device)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B")
tokenizer.pad_token = tokenizer.eos_token
print("Load tokenizer completed")


model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-3B", device_map="auto", torch_dtype="auto").to(device)

print("Load model completed")

Model device: cuda


/home/vasco/nguyehmk/miniconda3/envs/llm/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Load tokenizer completed


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.50s/it]


Load model completed


In [4]:
# Code block for question - answering

# messages = [
#     {"role": "user", "content": "Write a python function for calculating the factorial of an integer."},
# ]
# example = tokenizer.apply_chat_template(
# 	messages,
# 	add_generation_prompt=True,
# 	tokenize=True,
# 	return_dict=True,
# 	return_tensors="pt",
# ).to(device)

# with torch.inference_mode():
#     outputs = model.generate(**example, max_new_tokens=256)
# print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

## Forward function 
Captum uses forward function because it shows deterministic computation for prediction
whereas the method generate() is a high level abstract wrapper that computes multiples sequences of predictions

In [5]:
# Forward function 
# Captum uses this because it shows deterministic computation for prediction
# whereas generate() is a high level abstract method that computes multiples sequences of predictions

def forward_func(input_ids, attention_mask):

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    logits = outputs.logits

    next_token_logits = logits[:, -1, :]

    pred_token = next_token_logits.argmax(dim=-1)

    return next_token_logits.gather(
        1,
        pred_token.unsqueeze(1)
    ).squeeze(1)



In [6]:
# If do ablator = FeatureAblation(model)
# the feature ablation will use the whole model raw output to study => very big size tensor
# explain the whole model output (nonsense)

# Instead do ablator = FeatureAblation(forward_func)
# to choose what feature of the output to focus
# explain why model output this next token


fa = FeatureAblation(forward_func)

# attributions = fa.attribute(
#     inputs["input_ids"],
#     additional_forward_args=(
#         inputs["attention_mask"],
#     )
# )

## Visualization of token scores in a basic example prompt

At first, we test the feature ablation algorithm on a very simple python function, add(a, b), which returns the sum of two numbers.
We will visualize the scores using HTML colors, where blue represents positive attribution and red represents negative attribution. 

In this step, visualizing token scores helps understand how "important" a token is in the user's prompt. 


## Apply feature ablation on prompt with chat template

We test the method on prompt with chat template.

Chat template is a format to structure the conversation for LLM, it plays the role of a preprocessing step for the model's input prompt.

Chat template helps model distinguish between the user's messages and the system's ones.

In this beginning part, we will see if applying chat template affects the attribution scores given by feature ablation.


In [7]:
# Code for passing prompt and visualizing feature importance

messages = [
    {
        "role": "user",
        "content": """
Write some unit tests for this function:

def add(a, b):
    return a + b 
"""
    }
]

# Apply chat template to simplify communication with model
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

# Define the ablator to apply XAI method
attributions = fa.attribute(
    inputs["input_ids"],
    additional_forward_args=(
        inputs["attention_mask"],
    )
)


# Print out score for each token
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

clean_tokens = [
    tokenizer.convert_tokens_to_string([t])
    for t in tokens
]

scores = attributions.squeeze(0)

# print("===== Tokens as string =====\n")

# for t in clean_tokens:
#     print(t)


# print("===== Score of each token in the sequence =====\n")

# for clean_token, score in zip(clean_tokens, scores):
#     print(clean_token, float(score))


## Function to visualize token attribution

In [8]:
def print_code_scored(att_scores, att_tokens) : 
    att_scores = att_scores.cpu()
    norm_scores = scores / torch.max(torch.abs(scores))
    html = ""
    pairs_token_score = list(zip(att_tokens, norm_scores))

    span_init = f"""<span style=" background-color:"""
    span_mid = f"""; padding:2px; margin:1px; border-radius:3px; ">"""
    span_end = f"""</span> """
    line = ""

    for token, score in pairs_token_score:
        line += token + " "  
        
        if "\n" in token:
            #token.replace("\n", "<br>")
            token += "<br>"
            print("line", line) 
            line = ""
        if token == "   ":
            token = '&nbsp;&nbsp;&nbsp;'
    
        score = score.item()
    
        # positive -> red intensity
        intensity = int(255 * abs(score))

        if score > 0:
            color = f"rgba(0,0,255,{abs(score)})"
        else:
            color = f"rgba(255,0,0,{abs(score)})"
    
        html += f""" {span_init} {color}{span_mid} {token} {span_end}"""
    #print(html) 
    display(HTML(html))

## Function to sort attribution scores in increasing order

In [9]:
def print_code_sorted(att_scores, att_tokens):
    att_scores = att_scores.cpu()
    norm_scores = scores / torch.max(torch.abs(scores))
    html = ""
    pairs_token_score = list(zip(att_tokens, norm_scores))

    sorted_pairs = sorted(pairs_token_score, key=lambda x: x[1])

    sorted_seq = ""
    for token, score in sorted_pairs:
    
        score = score.item()
    
        # positive -> red intensity
        intensity = int(255 * abs(score))
    
        if score > 0:
            color = f"rgba(0,0,255,{abs(score)})"
        else:
            color = f"rgba(255,0,0,{abs(score)})"
    
        sorted_seq += f"""
        <span style="
            background-color:{color};
            padding:2px;
            margin:1px;
            border-radius:3px;
        ">
        {token}
        </span>
        """
    
    display(HTML(sorted_seq))

## Visualizing input features' importance

As can be seen from the sorted sequence, the most important tokens are the special tokens (chat delimiters <|im_start|>) inserted by the apply_chat_template() function.

With chat template applied, the model relies heavily on the formatting of the chat and it priorities the template tokens more than most of other tokens in the prompt. 

Since all of the tokens related to code and instruction are not important, removing (ablating) them will not affect the model's response much.

In [10]:
from IPython.display import display, HTML


scores = scores.cpu()
print_code_scored(scores, clean_tokens)

print("\n===== Tokens sorted by increasing importance =====\n")

print_code_sorted(scores, clean_tokens)



line <|im_start|> system 
 
line You  are  a  helpful  assistant . <|im_end|> 
 
line <|im_start|> user 

 
line Write  some  unit  tests  for  this  function :

 
line def  add (a ,  b ):
 
line      return  a  +  b  
 
line <|im_end|> 
 
line <|im_start|> assistant 
 



===== Tokens sorted by increasing importance =====



## Model's response with chat template

With chat template, the output shows how the real-world model behaves in practice.

We can see that the response is well formatted and easy to understand.

However, the trade-off is that we are not able to focus on the model's understanding of code semantics since the model pays more attention on template tokens and not code tokens.

In [11]:
# The output of model following the user's input

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Here are some unit tests for the `add` function:

```python
import unittest

def add(a, b):
    return a + b

class TestAdd(unittest.TestCase):
    def test_add_positive_numbers(self):
        self.assertEqual(add(1, 2), 3)

    def test_add_negative_numbers(self):
        self.assertEqual(add(-1, -2), -3)

    def test_add_zero(self):
        self.assertEqual(add(0, 0), 0)

    def test_add_mixed_numbers(self):
        self.assertEqual(add(1, -2), -1)

if __name__ == '__main__':
    unittest.main()
```

You can run these tests using the `unittest` module in Python. The `unittest` module provides a framework for writing unit tests in Python. The `unittest.TestCase` class is used to define test cases, and the `unittest.main()` function is used to run the tests.

The `test_add_positive_numbers` test case checks that the `add` function correctly adds two positive numbers. The `test_add_negative_numbers` test case checks that the `add` function correctly adds two negative numbers. The `tes

## Without chat template

Although all tokens have positive attribution, the results above are not interesting for studying how LLM understand code to write unit tests since most of the tokens related to code do not have high attribution scores.

So now we remove the chat template and keep the input prompt as text containing only the user's request and the code.

We simply remove roles and delimiter and keep the same content section of the prompt without any changes.

In [12]:
# Code for passing prompt and visualizing feature importance


message = """
Write some unit tests for this function:

def add(a, b):
    return a + b
"""

inputs = tokenizer(
    message,
    return_tensors="pt"
).to(device)

# Define the ablator to apply XAI method
attributions = fa.attribute(
    inputs["input_ids"],
    additional_forward_args=(
        inputs["attention_mask"],
    )
)


# Print out score for each token
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

clean_tokens = [
    tokenizer.convert_tokens_to_string([t])
    for t in tokens
]

scores = attributions.squeeze(0)


In [13]:
from IPython.display import display, HTML


scores = scores.cpu()
print_code_scored(scores, clean_tokens)

print("\n===== Tokens sorted by increasing importance =====\n")

print_code_sorted(scores, clean_tokens)


line 
 
line Write  some  unit  tests  for  this  function :

 
line def  add (a ,  b ):
 
line      return  a  +  b 
 



===== Tokens sorted by increasing importance =====



## Response of the model to the input

The visualization shows that the word 'tests' and the function's parameters were important in the prompt. Some of the instruction-related tokens have negative attribution.

The output of the block below shows the model's response to the prompt. The response is truncated as it is too long, and there is nothing beside the test function written by the model.

In comparison with the previous response, the scores are mostly negative to lowly positive as represented by the colors even though the response showed that the model understood and did the job.

From this output, we can understand why the template tokens are assigned with very high attribution scores. Without them, the quality of the response decreased strongly. The output is less organized, without any explicit explanations and the model does not even know when to properly stop the response.

In [14]:
# The output of model following the user's input

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.




def test_add():
    assert add(1, 1) == 2
    assert add(1, 2) == 3
    assert add(1, 3) == 4
    assert add(1, 4) == 5
    assert add(1, 5) == 6
    assert add(1, 6) == 7
    assert add(1, 7) == 8
    assert add(1, 8) == 9
    assert add(1, 9) == 10
    assert add(1, 10) == 11
    assert add(1, 11) == 12
    assert add(1, 12) == 13
    assert add(1, 13) == 14
    assert add(1, 14) == 15
    assert add(1, 15) == 16
    assert add(1, 16) == 17
    assert add(1, 17) == 18
    assert add(1, 18) == 


## Ablation test with chat template applied

We will see how the model performs with different level of ablation. We start from small ablation to heavy ablated prompt, with chat template applied.

### Small ablation
The function to be put in the prompt is

```
def add(a, b)
    return a  
```


In [15]:
# Code for passing prompt and visualizing feature importance


messages = [
    {
        "role": "user",
        "content": """
Write some unit tests for this function:

def add(a, b)
    return a 
"""
    }
]

# Apply chat template to simplify communication with model
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

# Define the ablator to apply XAI method
attributions = fa.attribute(
    inputs["input_ids"],
    additional_forward_args=(
        inputs["attention_mask"],
    )
)


# Print out score for each token
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

clean_tokens = [
    tokenizer.convert_tokens_to_string([t])
    for t in tokens
]


scores = attributions.squeeze(0)


In [16]:
from IPython.display import display, HTML


scores = scores.cpu()
print_code_scored(scores, clean_tokens)

print("\n===== Tokens sorted by increasing importance =====\n")

print_code_sorted(scores, clean_tokens)


line <|im_start|> system 
 
line You  are  a  helpful  assistant . <|im_end|> 
 
line <|im_start|> user 

 
line Write  some  unit  tests  for  this  function :

 
line def  add (a ,  b )
 
line      return  a  
 
line <|im_end|> 
 
line <|im_start|> assistant 
 



===== Tokens sorted by increasing importance =====



In [17]:
# The output of model following the user's input

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Certainly! Here are some unit tests for the `add` function:

```python
import unittest

def add(a, b):
    return a + b

class TestAddFunction(unittest.TestCase):
    def test_add_positive_numbers(self):
        self.assertEqual(add(2, 3), 5)

    def test_add_negative_numbers(self):
        self.assertEqual(add(-1, -1), -2)

    def test_add_zero(self):
        self.assertEqual(add(0, 0), 0)

    def test_add_mixed_numbers(self):
        self.assertEqual(add(5, -3), 2)

    def test_add_floats(self):
        self.assertAlmostEqual(add(1.5, 2.5), 4.0)

if __name__ == '__main__':
    unittest.main()
```

This test suite covers various scenarios including positive numbers, negative numbers, zero, mixed numbers, and floating-point numbers. Each test case uses the `assertEqual` or `assertAlmostEqual` methods to verify the correctness of the `add` function.<|endoftext|>


### Severe ablation

LLMs are often trained to have an excellent skill of auto-completing text. Taken the example above, if we remove some key tokens in the function such as '+' or 'return', the model still knows what to do.

This time, we push the model further by replacing the signature of the function as well as some other important tokens.

The function that we give to the model is:
```
def f(a, b)
    return a +
```


In [18]:
# Code for passing prompt and visualizing feature importance


messages = [
    {
        "role": "user",
        "content": """
Write some unit tests for this function:

def f(a, b)
    return a +
"""
    }
]

# Apply chat template to simplify communication with model
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

# Define the ablator to apply XAI method
attributions = fa.attribute(
    inputs["input_ids"],
    additional_forward_args=(
        inputs["attention_mask"],
    )
)


# Print out score for each token
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

clean_tokens = [
    tokenizer.convert_tokens_to_string([t])
    for t in tokens
]

scores = attributions.squeeze(0)

In [19]:
from IPython.display import display, HTML


scores = scores.cpu()
print_code_scored(scores, clean_tokens)

print("\n===== Tokens sorted by increasing importance =====\n")

print_code_sorted(scores, clean_tokens)


line <|im_start|> system 
 
line You  are  a  helpful  assistant . <|im_end|> 
 
line <|im_start|> user 

 
line Write  some  unit  tests  for  this  function :

 
line def  f (a ,  b )
 
line      return  a  +
 
line <|im_end|> 
 
line <|im_start|> assistant 
 



===== Tokens sorted by increasing importance =====



In [20]:
# The output of model following the user's input

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Here are some unit tests for the `f` function:

```python
import unittest

def f(a, b):
    return a + b

class TestF(unittest.TestCase):
    def test_f(self):
        self.assertEqual(f(1, 2), 3)
        self.assertEqual(f(5, 7), 12)
        self.assertEqual(f(-3, 4), 1)
        self.assertEqual(f(0, 0), 0)

if __name__ == '__main__':
    unittest.main()
```

You can run these tests using the `unittest` module in Python. The `unittest.TestCase` class provides methods for testing various aspects of your code, such as equality, inequality, and other assertions. The `assertEqual` method checks if two values are equal, and the `self.assertEqual` syntax is used to assert that the result of the `f` function is equal to the expected value.

You can add more test cases to the `TestF` class to cover different scenarios and edge cases. For example, you could test the function with negative numbers, floating-point numbers, or other types of inputs.<|endoftext|>


### Model's performance in severe ablation

From the result, the model is still able to complete the missing symbols in the given function and return the corresponding unit tests.

## Ablation test without chat template

This time we remove the chat template to see how the quality of the LLM's output change.

Since the quality of model's output is already low without chat template, we will only test it with very light ablation.

We expect to see low-quality responses that can still give some acceptable test cases from the model.

In [21]:
# Code for passing prompt and visualizing feature importance


message = """
Write some unit tests for this function:

def add(a, b)
     a + b
"""

inputs = tokenizer(
    message,
    return_tensors="pt"
).to(device)

# Define the ablator to apply XAI method
attributions = fa.attribute(
    inputs["input_ids"],
    additional_forward_args=(
        inputs["attention_mask"],
    )
)


# Print out score for each token
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

clean_tokens = [
    tokenizer.convert_tokens_to_string([t])
    for t in tokens
]

# print("===== Tokens as string =====\n")

# for t in clean_tokens:
#     print(t)

scores = attributions.squeeze(0)

# print("===== Score of each token in the sequence =====\n")

# for clean_token, score in zip(clean_tokens, scores):
#     print(clean_token, float(score))


In [22]:
from IPython.display import display, HTML


scores = scores.cpu()
print_code_scored(scores, clean_tokens)

print("\n===== Tokens sorted by increasing importance =====\n")

print_code_sorted(scores, clean_tokens)


line 
 
line Write  some  unit  tests  for  this  function :

 
line def  add (a ,  b )
 
line       a  +  b 
 



===== Tokens sorted by increasing importance =====



In [23]:
# The output of model following the user's input

with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



def subtract(a, b)
     a - b

def multiply(a, b)
     a * b

def divide(a, b)
     a / b

def power(a, b)
     a ** b

def mod(a, b)
     a % b

def add_mult(a, b, c)
     a + b + c * 3

def add_cubes(a, b)
     a ** 3 + b ** 3

def length_of_list(a)
     len(a)

def first_and_last(a)
     a[0] + a[-1]

def add_negatives(a)
     sum(x for x in a if x < 0)

def count_punctuation(a)
     sum(x in string.punctuation for x in a)

def count_punctuation2(a)
     count = 0
     for x in a:
         if x in string.punctuation:
             count += 1
     return count

def count_vowels(a)
     count = 0
     for x in a:
         if x in 'aeiouAEIOU':
             count += 1
     return count

def remove_vowels(a)
     return ''.join(x for x in a if x not in 'ae


### Result without chat template

We only removed the symbol ':' from the function but the model's response is chaotic.

This shows the role of chat template in the performance of LLMs.

## Feature ablation a more complicated function

In this part, we will redo the same experiment but with a function containing branches and loop.

We start with a python binary search.

In the prompt below, we added "Only give Python code and nothing else" at the end to because the model tend to "drift" with longer prompt.

In [24]:
# Code for passing prompt and visualizing feature importance


messages = [
    {
        "role": "user",
        "content": """
Write some unit tests for this function:

def binary_search(arr, x):
    low = 0
    high = len(arr) - 1

    while low <= high:
        mid = low + (high - low) // 2

        if arr[mid] < x:
            low = mid + 1
        elif arr[mid] > x:
            high = mid - 1
        else:
            return mid
    return -1

Only give Python code and nothing else.
"""
    }
]

# Apply chat template to simplify communication with model
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

# Define the ablator to apply XAI method
attributions = fa.attribute(
    inputs["input_ids"],
    additional_forward_args=(
        inputs["attention_mask"],
    )
)


# Print out score for each token
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

clean_tokens = [
    tokenizer.convert_tokens_to_string([t])
    for t in tokens
]

# print("===== Tokens as string =====\n")

# for t in clean_tokens:
#     print(t)

scores = attributions.squeeze(0)

# print("===== Score of each token in the sequence =====\n")

# for clean_token, score in zip(clean_tokens, scores):
#     print(clean_token, float(score))


In [25]:
from IPython.display import display, HTML


scores = scores.cpu()
print_code_scored(scores, clean_tokens)

print("\n===== Tokens sorted by increasing importance =====\n")

print_code_sorted(scores, clean_tokens)


line <|im_start|> system 
 
line You  are  a  helpful  assistant . <|im_end|> 
 
line <|im_start|> user 

 
line Write  some  unit  tests  for  this  function :

 
line def  binary _search (arr ,  x ):
 
line      low  =   0 
 
line      high  =  len (arr )  -   1 

 
line      while  low  <=  high :
 
line          mid  =  low  +  ( high  -  low )  //   2 

 
line          if  arr [mid ]  <  x :
 
line              low  =  mid  +   1 
 
line          elif  arr [mid ]  >  x :
 
line              high  =  mid  -   1 
 
line          else :
 
line              return  mid 
 
line      return  - 1 

 
line Only  give  Python  code  and  nothing  else .
 
line <|im_end|> 
 
line <|im_start|> assistant 
 



===== Tokens sorted by increasing importance =====



In [26]:
# The output of model following the user's input

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False,
        repetition_penalty=1.1
    )
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Here's an example of how you can write unit tests for the `binary_search` function in Python using the `unittest` module:<|fim_suffix|><|fim_middle|>

```python
import unittest

class TestBinarySearch(unittest.TestCase):

    def test_binary_search(self):
        # Test case 1: Search for element that exists in the list
        self.assertEqual(binary_search([1, 3, 5, 7, 9], 5), 2)

        # Test case 2: Search for element that does not exist in the list
        self.assertEqual(binary_search([1, 3, 5, 7, 9], 4), -1)

        # Test case 3: Search for element at the beginning of the list
        self.assertEqual(binary_search([1, 3, 5, 7, 9], 1), 0)

        # Test case 4: Search for element at the end of the list
        self.assertEqual(binary_search([1, 3, 5, 7, 9], 9), 4)

if __name__ == '__main__':
    unittest.main()
```

In this example, we create a class called `TestBinarySearch` which inherits from `unittest.TestCase`. We define four test cases within this class to cover diff

### Binary search generated tests

The python tests given by the model is acceptable as it covers all branches in the function.

The response also gave clear instructions and comments to help users understand eventhough there is the "Only give Python code and nothing else." in the prompt.

This can be explained by the negative importance of tokens in the final instruction part.

## Light ablation

We remove some key symbols in the function such as 'while', 'else', some arithmetic operations and alternate the function signature to see how the model answers in this case.

In [27]:
# Code for passing prompt and visualizing feature importance


messages = [
    {
        "role": "user",
        "content": """
Write some unit tests for this function:

def f(arr, x):
    low = 0
    high = len(arr) - 1

     low <= high:
        mid = low + (high - low) // 2

        if arr[mid] < x:
            low = mid  1
        elif arr[mid] > x:
            high = mid  1
        :
            return mid
    return -1

Only give Python code and nothing else.
"""
    }
]

# Apply chat template to simplify communication with model
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

# Define the ablator to apply XAI method
attributions = fa.attribute(
    inputs["input_ids"],
    additional_forward_args=(
        inputs["attention_mask"],
    )
)


# Print out score for each token
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

clean_tokens = [
    tokenizer.convert_tokens_to_string([t])
    for t in tokens
]

# print("===== Tokens as string =====\n")

# for t in clean_tokens:
#     print(t)

scores = attributions.squeeze(0)

# print("===== Score of each token in the sequence =====\n")

# for clean_token, score in zip(clean_tokens, scores):
#     print(clean_token, float(score))


In [28]:
from IPython.display import display, HTML


scores = scores.cpu()
print_code_scored(scores, clean_tokens)

print("\n===== Tokens sorted by increasing importance =====\n")

print_code_sorted(scores, clean_tokens)


line <|im_start|> system 
 
line You  are  a  helpful  assistant . <|im_end|> 
 
line <|im_start|> user 

 
line Write  some  unit  tests  for  this  function :

 
line def  f (arr ,  x ):
 
line      low  =   0 
 
line      high  =  len (arr )  -   1 

 
line       low  <=  high :
 
line          mid  =  low  +  ( high  -  low )  //   2 

 
line          if  arr [mid ]  <  x :
 
line              low  =  mid     1 
 
line          elif  arr [mid ]  >  x :
 
line              high  =  mid     1 
 
line          :
 
line              return  mid 
 
line      return  - 1 

 
line Only  give  Python  code  and  nothing  else .
 
line <|im_end|> 
 
line <|im_start|> assistant 
 



===== Tokens sorted by increasing importance =====



In [29]:
# The output of model following the user's input

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False,
        repetition_penalty=1.1
    )
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Here's an example of how you could write unit tests for the `f` function in Python using the `unittest` module:<|fim_suffix|><|fim_middle|>

```python
import unittest

class TestF(unittest.TestCase):

    def test_f(self):
        self.assertEqual(f([1, 3, 5, 7], 4), -1)
        self.assertEqual(f([1, 3, 5, 7], 5), 2)

if __name__ == '__main__':
    unittest.main()
```

In this example, we define a class called `TestF` that inherits from `unittest.TestCase`. We then define two test methods, `test_f`, which checks whether the function returns the expected result when given different inputs.

To run these tests, simply execute the script containing them. The output will indicate whether all tests passed or failed.<|endoftext|>


### Result from light ablation

The appearance of tokens like <|fim_suffix|>, <|fim_middle|> and <|endoftext|> suggests that the model faced a small drift in its thinking process.

The tests in the response are incomplete because the function code is ablated. The model chosed to write tests to cover branches that are unchanged in the code.